# OpenAlex corpus

## Setup

In [72]:
from pathlib import Path
import os
import math
import duckdb
import pandas as pd
import requests
import yaml
from dotenv import load_dotenv

def get_config(path):
    path = Path(path)
    with path.open(encoding="utf-8") as file:
        config = yaml.safe_load(file)
    return config

config = get_config("../config/config.yaml")
corpus = config["corpus"]

load_dotenv()
config["api_key"] = os.environ["OPENALEX_API_KEY"]

## Query

In [73]:
works_url = "https://api.openalex.org/works"

subfields = "|".join(str(subfield) for subfield in corpus["subfields"])

filters = ",".join([
    f"publication_year:>{corpus['min_year'] - 1}",  # filter is strict >, so -1 keeps min_year in
    f"has_abstract:{str(corpus['has_abstract']).lower()}",
    f"primary_topic.subfield.id:{subfields}",
])

params = {
    "filter": filters,
    "select": ",".join(corpus["cols_to_select"]),
    "api_key": config["api_key"],
}

## Helpers

In [74]:
def paginate(url, params=None, per_page=200):
    """Yield successive pages from a cursor-paginated OpenAlex endpoint."""
    params = dict(params or {})
    params["per-page"] = per_page
    cursor = "*"

    while cursor:
        params["cursor"] = cursor
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        page = response.json()

        if not page.get("results"):
            break

        yield page
        cursor = page.get("meta", {}).get("next_cursor")

In [75]:
def reconstruct_abstract(inverted_abstract):
    if not inverted_abstract:
        return None, 0

    abstract = []
    for word, positions in inverted_abstract.items():
        for position in positions:
            while len(abstract) <= position:
                abstract.append(None)
            abstract[position] = word

    n_gaps = abstract.count(None)
    text = " ".join([word for word in abstract if word is not None])
    return text, n_gaps

In [76]:
assert reconstruct_abstract({"The": [0], "cat": [1], "sat": [2]}) == ("The cat sat", 0)
assert reconstruct_abstract({"The": [0], "cat": [1], "sat": [3]}) == ("The cat sat", 1)
assert reconstruct_abstract(None) == (None, 0)
assert reconstruct_abstract({}) == (None, 0)

In [77]:
def remove_prefix(item):
    if item is None:
        return None

    value = item.removeprefix("https://doi.org/")
    value = value.removeprefix("https://openalex.org/subfields/")
    value = value.removeprefix("https://openalex.org/")

    return value

In [78]:
def flatten_work(work):
    work_id = remove_prefix(work['id'])
    abstract, gap_count = reconstruct_abstract(work['abstract_inverted_index'])

    work_row = {'work_id': work_id,
                'doi': remove_prefix(work['doi']),
                'title': work['title'],
                'abstract': abstract,
                'abstract_gap_count': gap_count,
                'publication_year': work['publication_year'],
                'publication_date': work['publication_date'],
                'cited_by_count': work['cited_by_count'],
                'subfield_id': int(remove_prefix(work['primary_topic']['subfield']['id'])),
                'subfield_name': work['primary_topic']['subfield']['display_name'],
                'topic_id': remove_prefix(work['primary_topic']['id']),
                'topic_name': work['primary_topic']['display_name'],
                'language': work['language']
                }

    author_rows = []
    for author_order, authorship in enumerate(work['authorships'], start=1):
        author_row = {'work_id': work_id,
                      'author_order': author_order,
                      'author_id': remove_prefix(authorship['author']['id']),
                      'author_name': authorship['author']['display_name']
                      }
        author_rows.append(author_row)

    reference_rows = []
    for url in work['referenced_works']:
        reference_row = {'citing_work_id': work_id, 'cited_work_id': remove_prefix(url)}
        reference_rows.append(reference_row)

    return (work_row, author_rows, reference_rows)

## Sanity checks on one page

In [79]:
pages = paginate(works_url, params, per_page=25)
first_page = next(pages)
print("Number of results:", first_page["meta"]["count"])

Number of results: 2653801


In [80]:
for work in first_page["results"]:
    print(work["title"])

lmerTest Package: Tests in Linear Mixed Effects Models
Applied Multiple Regression/Correlation Analysis for the Behavioral Sciences
Fractional Differential Equations
Investigation of the freely available easy-to-use software ‘EZR’ for medical statistics
UCHIME improves sensitivity and speed of chimera detection
Google Earth Engine: Planetary-scale geospatial analysis for everyone
An Introduction to Propensity Score Methods for Reducing the Effects of Confounding in Observational Studies
emcee : The MCMC Hammer
Köppen's climate classification map for Brazil
- STATISTICAL PACKAGE FOR SOCIAL SCIENCES
Food Security: The Challenge of Feeding 9 Billion People
A Mathematical Theory of Evidence
Multivariate data analysis : a global perspective
Collinearity: a review of methods to deal with it and a simulation study evaluating their performance
Correlation Coefficients: Appropriate Use and Interpretation
Applied Multivariate Statistics for the Social Sciences
A general and simple method for obt

In [81]:
print(first_page["results"][0].keys())

dict_keys(['id', 'doi', 'title', 'authorships', 'cited_by_count', 'publication_year', 'publication_date', 'primary_topic', 'referenced_works', 'abstract_inverted_index', 'language'])


In [82]:
[reconstruct_abstract(work["abstract_inverted_index"]) for work in first_page["results"][:5]]
# All abstracts read normally, seems to work okay

[("One of the frequent questions by users of the mixed model function lmer of the lme4 package has been: How can I get p values for the F and t tests for objects returned by lmer? The lmerTest package extends the 'lmerMod' class of the lme4 package, by overloading the anova and summary functions by providing p values for tests for fixed effects. We have implemented the Satterthwaite's method for approximating degrees of freedom for the t and F tests. We have also implemented the construction of Type I - III ANOVA tables. Furthermore, one may also obtain the summary as well as the anova table using the Kenward-Roger approximation for denominator degrees of freedom (based on the KRmodcomp function from the pbkrtest package). Some other convenient mixed model analysis tools such as a step method, that performs backward elimination of nonsignificant effects - both random and fixed, calculation of population means and multiple comparison tests together with plot facilities are provided by t

## Database

In [83]:
con = duckdb.connect(corpus["db_path"])

In [84]:
con.execute("""CREATE TABLE IF NOT EXISTS works (
                    work_id VARCHAR,
                    doi VARCHAR,
                    title VARCHAR,
                    abstract VARCHAR,
                    abstract_gap_count INTEGER,
                    publication_year INTEGER,
                    publication_date DATE,
                    cited_by_count INTEGER,
                    subfield_id INTEGER,
                    subfield_name VARCHAR,
                    topic_id VARCHAR,
                    topic_name VARCHAR,
                    language VARCHAR,
                    PRIMARY KEY (work_id)
                    )""")

con.execute("""CREATE TABLE IF NOT EXISTS work_authors (
                    work_id VARCHAR,
                    author_order INTEGER,
                    author_id VARCHAR,
                    author_name VARCHAR,
                    PRIMARY KEY (work_id, author_order)
                    )""")

con.execute("""CREATE TABLE IF NOT EXISTS work_references (
                    citing_work_id VARCHAR,
                    cited_work_id VARCHAR
                    )""")

In [85]:
print(con.sql("SHOW TABLES"))
print(con.sql("DESCRIBE works"))
print(con.sql("DESCRIBE work_authors"))
print(con.sql("DESCRIBE work_references"))

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ work_authors    │
│ work_references │
│ works           │
└─────────────────┘

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ work_id            │ VARCHAR     │ NO      │ PRI     │ NULL    │ NULL    │
│ doi                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ title              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ abstract           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ abstract_gap_count │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ publication_year   │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ publication_date   │ DATE        │ YES     │ NULL    │ NULL    │ NU

## Check flatten_work against the tables

In [86]:
work_columns = set(con.sql("DESCRIBE works").df()["column_name"])
author_columns = set(con.sql("DESCRIBE work_authors").df()["column_name"])
reference_columns = set(con.sql("DESCRIBE work_references").df()["column_name"])

n_authors = 0
n_references = 0
n_null_author_ids = 0

for work in first_page["results"]:
    work_row, author_rows, reference_rows = flatten_work(work)
    work_id = work_row["work_id"]

    # The work row has exactly the columns of the works table
    assert set(work_row) == work_columns, work_id

    # One author row per authorship, numbered from 1, all pointing back to this work
    assert len(author_rows) == len(work["authorships"]), work_id
    for row in author_rows:
        assert set(row) == author_columns, work_id
        assert row["work_id"] == work_id, work_id
    if author_rows:
        assert author_rows[0]["author_order"] == 1, work_id

    # One reference row per referenced work, all pointing back to this work
    assert len(reference_rows) == len(work["referenced_works"]), work_id
    for row in reference_rows:
        assert set(row) == reference_columns, work_id
        assert row["citing_work_id"] == work_id, work_id

    n_authors += len(author_rows)
    n_references += len(reference_rows)
    n_null_author_ids += sum(row["author_id"] is None for row in author_rows)

print(f"All checks passed: {len(first_page['results'])} works, {n_authors} author rows, "
      f"{n_references} reference rows, {n_null_author_ids} author rows with no author ID")

All checks passed: 25 works, 99 author rows, 1473 reference rows, 7 author rows with no author ID


# Step 5

In [87]:
def year_params(year):
    new_params = params.copy()
    new_params['filter'] = ",".join([
        f"publication_year:{year}",
        f"has_abstract:{str(corpus['has_abstract']).lower()}",
        f"primary_topic.subfield.id:{subfields}",
])
    return new_params

year_params(2010)['filter']

'publication_year:2010,has_abstract:true,primary_topic.subfield.id:2302|2303|2309|1105|2611|2613'

In [88]:
def insert_rows(table, rows):
    if not rows:
        print(f"No rows for {table} on this page, skipping.")
        return

    rows_df = pd.DataFrame(rows)
    con.execute(f"INSERT INTO {table} BY NAME SELECT * FROM rows_df")


def ingest_year(year):
    con.begin()
    try:
        pages = paginate(works_url, year_params(year), per_page=corpus['per_page'])
        for page_number, page in enumerate(pages, start=1):
            if page_number == 1:
                n_works = page["meta"]["count"]
                n_pages = math.ceil(n_works / corpus["per_page"])
                print(f"{n_works} works in {year}, {n_pages} pages to fetch.")

            works_list = []
            authors_list = []
            references_list = []

            for work in page['results']:
                work_row, work_authors, work_references = flatten_work(work)
                
                works_list.append(work_row)
                authors_list.extend(work_authors)
                references_list.extend(work_references)
                
            insert_rows("works", works_list)
            insert_rows("work_authors", authors_list)
            insert_rows("work_references", references_list)

            if page_number % 50 == 0: print(page_number, "pages parsed.")
        con.commit()

    except BaseException:
        con.rollback()
        raise

In [89]:
ingest_year(2010)

93801 works in 2010, 470 pages to fetch.
50 pages parsed.
100 pages parsed.
150 pages parsed.
200 pages parsed.
250 pages parsed.
300 pages parsed.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
350 pages parsed.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No rows for work_references on this page, skipping.
No

In [90]:
checks = {
    "Works (should equal the API count, 93801)":
        "SELECT count(*) AS n_works FROM works",
    "Years (should be 2010 only)":
        "SELECT publication_year, count(*) AS n FROM works GROUP BY publication_year",
    "Shares on works (sample: ~0.15 null DOI, ~0.58 no references, ~0.01 gaps)":
        """SELECT
               avg((doi IS NULL)::INTEGER) AS null_doi_share,
               avg((work_id NOT IN (SELECT citing_work_id FROM work_references))::INTEGER) AS no_reference_share,
               avg((abstract_gap_count > 0)::INTEGER) AS gap_share
           FROM works""",
    "Author rows with no author ID (sample: ~0.11)":
        "SELECT avg((author_id IS NULL)::INTEGER) AS null_author_id_share FROM work_authors",
    "Duplicate reference pairs (should be 0)":
        """SELECT count(*) AS duplicate_pairs
           FROM (SELECT citing_work_id, cited_work_id
                 FROM work_references
                 GROUP BY citing_work_id, cited_work_id
                 HAVING count(*) > 1)""",
    "Works per subfield":
        "SELECT subfield_name, count(*) AS n FROM works GROUP BY subfield_name ORDER BY n DESC",
    "Most common topics":
        "SELECT topic_name, count(*) AS n FROM works GROUP BY topic_name ORDER BY n DESC LIMIT 15",
    "Works in 'Species Distribution and Climate Change' (T10895)":
        "SELECT count(*) AS n FROM works WHERE topic_id = 'T10895'",
}

for label, query in checks.items():
    print(label)
    con.sql(query).show()


Works (should equal the API count, 93801)
┌─────────┐
│ n_works │
│  int64  │
├─────────┤
│   93801 │
└─────────┘

Years (should be 2010 only)
┌──────────────────┬───────┐
│ publication_year │   n   │
│      int32       │ int64 │
├──────────────────┼───────┤
│             2010 │ 93801 │
└──────────────────┴───────┘

Shares on works (sample: ~0.15 null DOI, ~0.58 no references, ~0.01 gaps)
┌────────────────────┬────────────────────┬──────────────────────┐
│   null_doi_share   │ no_reference_share │      gap_share       │
│       double       │       double       │        double        │
├────────────────────┼────────────────────┼──────────────────────┤
│ 0.4361680579098304 │ 0.5303354974893658 │ 0.020564812741868423 │
└────────────────────┴────────────────────┴──────────────────────┘

Author rows with no author ID (sample: ~0.11)
┌──────────────────────┐
│ null_author_id_share │
│        double        │
├──────────────────────┤
│  0.04780053531944606 │
└──────────────────────┘

Duplicat

In [91]:
print("--- Random sample ---")
for title, abstract in con.sql("SELECT title, abstract FROM works USING SAMPLE 5").fetchall():
    print(title, "\n", abstract, "\n")

print("--- Shortest abstracts ---")
for title, abstract, n_chars in con.sql(
        "SELECT title, abstract, length(abstract) AS n_chars FROM works ORDER BY n_chars LIMIT 5").fetchall():
    print(n_chars, "|", title, "|", abstract)


--- Random sample ---
Sampling gap fraction and size for estimating leaf area and clumping indices from hemispherical photographs 
 Hemispherical photography is becoming a popular technique for gap fraction measurements to characterize biophysical parameters and solar radiation in plant canopies. One of the crucial steps in the measurement of canopy gap fraction using hemispherical photography is determining the resolution of the sampling grid. In this work, the effects of varying resolutions of sampling grids by modifying the angle widths of zenithal annuli and azimuthal sectors were evaluated for leaf area and clumping indices computations. Sensitivity analysis was performed to test these effects using artificial photographs simulating ideal canopies with varying leaf area index and aggregation levels of foliage elements. Contrasting forest types, including natural tropical cloud forest and exotic plantations, were tested as real canopies. Results indicate that leaf area and clumping

1 | Some observations on nesting African Grey Parrots, Psittacus erithacus, in Uganda | -
1 | Presence-only data, pseudo-absences, and other lies about habitat selection | .
1 | Uso de manantiales de filtración por los vertebrados durante la época seca en un bosque tropical fragmentado en la costa de Michoacán. | -
1 | Further Records of Stripe-backed Weasel Mustela strigidorsa from Lao PDR | -
1 | Plants from mining wastes from Taxco, Guerrero, Mexico | -
